# Phase 1 Training Notebook: Event Extraction

This notebook provides a runnable training scaffold for Stage 1 (`EventTypeClassifier`, `ArgumentExtractor`, `MagnitudeRegressor`) and visualizes training dynamics with seaborn.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import seaborn as sns
import matplotlib.pyplot as plt

# Resolve DirectionA as project root so `from src...` imports work from src/training.
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.phase1.event_extractor import (
    EventTypeClassifier,
    ArgumentExtractor,
    MagnitudeRegressor,
    EventExtractionLoss,
    NUM_EVENT_TYPES,
)

sns.set_theme(style='whitegrid', context='talk')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
hidden_size = 768
seq_len = 48
batch_size = 16
steps_per_epoch = 20
epochs = 8

type_head = EventTypeClassifier(hidden_size=hidden_size).to(device)
arg_head = ArgumentExtractor(hidden_size=hidden_size).to(device)
mag_head = MagnitudeRegressor(hidden_size=hidden_size).to(device)
criterion = EventExtractionLoss(lambda_type=1.0, lambda_arg=1.0, lambda_mag=0.5)

optimizer = torch.optim.AdamW(
    list(type_head.parameters()) + list(arg_head.parameters()) + list(mag_head.parameters()),
    lr=2e-4,
    weight_decay=1e-2,
)

In [ ]:
history = []

for epoch in range(1, epochs + 1):
    running = {'total': 0.0, 'type': 0.0, 'arg': 0.0, 'magnitude': 0.0, 'acc': 0.0}

    for _ in range(steps_per_epoch):
        sequence_hidden = torch.randn(batch_size, seq_len, hidden_size, device=device)
        cls_hidden = sequence_hidden[:, 0, :]
        attention_mask = torch.ones(batch_size, seq_len, device=device, dtype=torch.long)

        y_type = torch.randint(0, NUM_EVENT_TYPES, (batch_size,), device=device)
        y_sub_s = torch.randint(0, seq_len, (batch_size,), device=device)
        y_sub_e = torch.maximum(y_sub_s, torch.randint(0, seq_len, (batch_size,), device=device))
        y_obj_s = torch.randint(0, seq_len, (batch_size,), device=device)
        y_obj_e = torch.maximum(y_obj_s, torch.randint(0, seq_len, (batch_size,), device=device))
        y_mag = torch.randn(batch_size, device=device)
        y_mag_mask = torch.rand(batch_size, device=device) > 0.1

        pred_type = type_head(cls_hidden)
        start_logits, end_logits = arg_head(sequence_hidden, attention_mask)
        pred_mag = mag_head(cls_hidden, y_type)

        preds = {
            'event_type_logits': pred_type,
            'start_logits': start_logits,
            'end_logits': end_logits,
            'magnitude': pred_mag,
        }
        targets = {
            'event_type_labels': y_type,
            'subject_start': y_sub_s,
            'subject_end': y_sub_e,
            'object_start': y_obj_s,
            'object_end': y_obj_e,
            'magnitude_labels': y_mag,
            'magnitude_mask': y_mag_mask,
        }

        losses = criterion(preds, targets)
        arg_loss = (losses['subject_start'] + losses['subject_end'] + losses['object_start'] + losses['object_end']) / 4

        optimizer.zero_grad(set_to_none=True)
        losses['total'].backward()
        nn.utils.clip_grad_norm_(list(type_head.parameters()) + list(arg_head.parameters()) + list(mag_head.parameters()), 1.0)
        optimizer.step()

        acc = (pred_type.argmax(dim=-1) == y_type).float().mean()
        running['total'] += losses['total'].item()
        running['type'] += losses['type'].item()
        running['arg'] += arg_loss.item()
        running['magnitude'] += losses['magnitude'].item()
        running['acc'] += acc.item()

    for k in running:
        running[k] /= steps_per_epoch
    running['epoch'] = epoch
    history.append(running)

history_df = pd.DataFrame(history)
history_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

long_loss = history_df.melt(id_vars='epoch', value_vars=['total', 'type', 'arg', 'magnitude'], var_name='loss_name', value_name='value')
sns.lineplot(data=long_loss, x='epoch', y='value', hue='loss_name', marker='o', ax=axes[0])
axes[0].set_title('Phase 1 Training Losses')

sns.lineplot(data=history_df, x='epoch', y='acc', marker='o', color='tab:green', ax=axes[1])
axes[1].set_ylim(0, 1)
axes[1].set_title('Event Type Accuracy (Synthetic)')

plt.tight_layout()
plt.show()

## Switching to Real Data
1. Replace the synthetic tensors with tokenized financial news batches.
2. Use `FinBERTEventExtractor` for full end-to-end training instead of isolated heads.
3. Keep the same logging schema to reuse seaborn plots for real experiments.